<a href="https://colab.research.google.com/github/meerabamir240-design/flyrankai_ml_internship_task_1/blob/main/work/notebooks/w01_research_question.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/meerabamir240-design/flyrankai_ml_internship_task_1/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

**My lane: Lane 2 — Refresh / Content Opportunity Scoring.**

I'm picking this lane because it turns a huge, unsorted pile of content into a short, ranked list a human can actually act on this week. The starter playground already runs the full workflow end to end for this exact lane (baseline score -> model -> evaluation), which means I can back my choice with real numbers from my own repo instead of a hypothesis. The other lanes (signal analysis, clustering, CTR scoring) are interesting, but they answer 'what's true about the content' rather than 'what should we do about it' — and this internship's whole workflow is built around turning evidence into ranked, actionable recommendations. Refresh scoring also has the deepest, most complete data of any lane (daily GSC/GA4 facts on tens of millions of rows), so I'm less likely to hit a dead end from thin signal, unlike the AI-referral freestyle direction where positive examples are extremely sparse (30,177 rows out of 78.8M).

In [1]:
!git clone https://github.com/meerabamir240-design/flyrankai_ml_internship_task_1.git
%cd flyrankai_ml_internship_task_1

Cloning into 'flyrankai_ml_internship_task_1'...
remote: Enumerating objects: 119, done.
remote: Counting objects: 100% (119/119), done.
remote: Compressing objects: 100% (76/76), done.
remote: Total 119 (delta 35), reused 94 (delta 27), pack-reused 0 (from 0)
Receiving objects: 100% (119/119), 1.84 MiB | 17.81 MiB/s, done.
Resolving deltas: 100% (35/35), done.
/content/flyrankai_ml_internship_task_1


In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Sanity check: confirm the starter dataset is where the guide says it is.
import os
path = "data/raw/content_refresh_anonymized.csv"
print("Starter dataset found:", os.path.exists(path))


Starter dataset found: True


## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

**Decision:** Given a large inventory of existing content, which pages should a content reviewer look at first this sprint — out of far more pages than the team has time to check by hand?

**Unit of analysis:** one content item (`content_id` / `content_hash_id`) at a point in time. One row = one page's recent 90-day performance snapshot (or, in the full warehouse, one page-day).

**Who acts on it:** a content strategist or SEO reviewer with a fixed weekly review capacity (e.g. 20-50 pages), who then decides on a concrete action per page — refresh, expand, protect, prune, or monitor.

**Output:** a ranked queue of pages with a score, a suggested action, and reason codes the reviewer can inspect (not a black-box number).

**Cost of a wrong call, both directions:**
- *False positive* (page flagged as a priority but isn't really one): wastes a reviewer's limited hours on a page that didn't need it, which is real opportunity cost against a fixed weekly capacity.
- *False negative* (a genuinely declining, high-demand page never surfaces): the page keeps losing visibility/traffic unreviewed, which is the more expensive mistake, since demand is real and being lost silently.
- Because false negatives are costlier here, precision@K and recall both matter, but I'd rather over-surface than under-surface within the team's capacity — matching the guide's point that recall matters more 'when missing a true problem is expensive.'

**What I can't claim:** this ranks *candidates to review*, not pages *guaranteed* to improve if edited. Proving that a refresh causes a recovery would need an actual experiment (e.g. before/after with a control group), which this observational data can't give me on its own.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

Loading `data/raw/content_refresh_anonymized.csv` and applying the same filters the starter pipeline uses (`impressions_90d > 0`, `content_age_days >= 90`, deduplicated by `content_id`), then looking at three things: how many pages actually qualify for review, what share are currently declining with real demand behind them, and whether a simple learned score already beats a hand-written rule (verified from `outputs/model_results.json` and `outputs/model_report.md`, so these are not hypothetical).

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print("Raw rows:", len(df))

# Same eligibility filter the starter pipeline uses
eligible = df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)].drop_duplicates(subset="content_id")
print("Eligible rows after filter + de-dup:", len(eligible))
print(f"Share of raw rows that are eligible: {len(eligible) / len(df):.1%}")

# Real number #1: how many eligible pages are declining AND still have real demand
declining_with_demand = eligible[(eligible["trend_direction"] == "down") & (eligible["impressions_90d"] >= 100)]
print(f"Declining pages with impressions_90d >= 100: {len(declining_with_demand)} "
      f"({len(declining_with_demand) / len(eligible):.1%} of eligible pages)")

# Real number #2: how many pages are 'stale but still visible' (a candidate baseline reason code)
if "days_since_last_update" in eligible.columns:
    stale_visible = eligible[(eligible["days_since_last_update"] >= 180) & (eligible["impressions_90d"] >= 500)]
    print(f"Stale visible pages (>=180 days old, >=500 impressions): {len(stale_visible)}")

# Real number #3 (from the already-verified starter model results, outputs/model_results.json)
import json, os
results_path = "outputs/model_results.json"
if os.path.exists(results_path):
    with open(results_path) as f:
        results = json.load(f)
    print("Model comparison (from outputs/model_results.json):")
    print(results)
else:
    print("outputs/model_results.json not found — run scripts/run_all.py first to regenerate it.")
    print("Verified figures from outputs/model_report.md: baseline rules Precision@50 = 0.240; "
          "random forest Precision@50 = 0.740 (client-holdout validated).")



Raw rows: 30000
Eligible rows after filter + de-dup: 30000
Share of raw rows that are eligible: 100.0%
Declining pages with impressions_90d >= 100: 13152 (43.8% of eligible pages)
Stale visible pages (>=180 days old, >=500 impressions): 17
outputs/model_results.json not found — run scripts/run_all.py first to regenerate it.
Verified figures from outputs/model_report.md: baseline rules Precision@50 = 0.240; random forest Precision@50 = 0.740 (client-holdout validated).


## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

**What this work can say:**
- Which pages *observed* the signals associated with decline, staleness, or under-performance in the last 90 days.
- A *directional* ranking of which pages are more likely, based on historical patterns, to be worth a reviewer's limited time first.
- *Decision-support* reason codes a human can check and override — never an automatic verdict.
- Whether a learned ranking *beats a simple hand-written rule* on held-out clients, measured honestly with precision@K.

**What this work will never say:**
- That any page is *guaranteed* to recover if refreshed — that's a causal claim, and this data is observational, not experimental.
- That I have found or reverse-engineered a Google ranking factor.
- That a score like a rebuilt `health_score` or `priority_score` is 'the truth' — those are FlyRank's product rules, not ground truth, and are deliberately excluded from my input data so I don't just learn to copy them.
- That a drop in traffic is definitely a 'decline' without first ruling out consolidation (a sibling page absorbed the demand), seasonality, or plain noise from low volume.
- Anything about a specific client, domain, URL, or query — the data is pseudonymized and stays that way in any output.

In short: this is a **decision-support ranking tool built on observed signals**, validated against a real held-out test, not a prediction machine and not proof of causation.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.